# NRP-Hosted LLMs: Access Setup and a First API Call
### Chris Endemann, endemann@wisc.edu
### [Nexus version](https://uw-madison-datascience.github.io/ML-X-Nexus/Learn/Notebooks/NRP-Hosted-LLMs.html)
### Categories
- Notebooks
- Code-along
- LLM
- NRP
- API


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-DataScience/ML-X-Nexus/blob/main/Learn/Notebooks/NRP-Hosted-LLMs.ipynb)

**Free frontier-LLM endpoints for researchers.** The [National Research Platform (NRP)](https://nrp.ai/) hosts open-weight models (Qwen3, GPT-OSS, Kimi, Gemma, embeddings, etc.) behind an OpenAI-compatible API at **no cost** to researchers and educators at [InCommon](https://incommon.org/)-affiliated institutions — UW–Madison included. This notebook walks through getting an access token and making your first calls (chat, reasoning, multimodal, embeddings). For the fuller reference — fair-use limits, cluster policies, where it fits alongside CHTC / UW Cloud — see the [NRP card](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/NRP-Nautilus.html).

::: {.callout-warning}
You need your own NRP token to run these cells. The notebook is rendered as-is — without a token, every call returns `401 Unauthorized`.
:::

## 1. Get access

1. **Sign in** at [nrp.ai](https://nrp.ai/) via CILogon. Pick one identity provider and stick with it — switching IdPs later fails with a "Permission denied" error.
2. **Get into an LLM-enabled namespace.** Already in a namespace? Check at [nrp.ai/namespaces](https://nrp.ai/namespaces) (LLM-enabled ones are flagged). Need a new namespace or the LLM flag added? Ask in the **Nautilus Support** channel in [Matrix](https://nrp.ai/contact/) or email [support@nationalresearchplatform.org](mailto:support@nationalresearchplatform.org). See the [NRP card](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/NRP-Nautilus.html) for the three signup patterns (solo, lab, student).
3. **Generate a token** at [nrp.ai/llmtoken](https://nrp.ai/llmtoken).

## 2. Configure the client

Install `openai` and point it at NRP's endpoint. Store the token in an env var (e.g., a local `.env`), not in code.

In [ ]:
# %pip install --quiet openai python-dotenv requests

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(
    api_key=os.environ["NRP_LLM_TOKEN"],
    base_url="https://ellm.nrp-nautilus.io/v1",
)

## 3. List available models

In [ ]:
for m in client.models.list().data:
    print(m.id)

Expect names like `qwen3`, `qwen3-small`, `gpt-oss`, `gemma`, `kimi`, `glm-5`, `qwen3-embedding`. The exact set rotates — see the [lifecycle & changelog](https://nrp.ai/documentation/userdocs/ai/llm-managed/lifecycle/).

## 4. A chat completion

`gpt-oss` is a good first call — text-only, low GPU footprint, stable across model updates.

In [ ]:
completion = client.chat.completions.create(
    model="gpt-oss",
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant for a graduate ML course."},
        {"role": "user", "content": "In 2-3 sentences, when should I use LoRA instead of full fine-tuning?"},
    ],
)
print(completion.choices[0].message.content)

::: {.callout-caution}
If your client needs a `max_tokens` value, set it to ~1/3–1/4 of the model's context — it caps output, not total context.
:::

## 5. Toggle reasoning mode

Several models default to "thinking" mode, which adds latency. Disable it via `extra_body`:

In [ ]:
completion = client.chat.completions.create(
    model="qwen3-small",
    messages=[{"role": "user", "content": "One-line definition of overfitting."}],
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(completion.choices[0].message.content)

Exact keys vary by model — see each [model card](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) for `glm-5`, `kimi`, etc.

## 6. Multimodal: image input

`gemma`, `qwen3`, `qwen3-small`, `gemma-small`, and `kimi` accept image inputs as `image_url` content blocks (URL or base64 data URI):

In [ ]:
import base64, requests

img = requests.get("https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg", timeout=30).content
b64 = base64.b64encode(img).decode("ascii")

completion = client.chat.completions.create(
    model="gemma",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "In one sentence: what animal and what is it doing?"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ],
    }],
)
print(completion.choices[0].message.content)

`gemma-small` is the only catalogued model that also accepts **audio** input (ASR / speech-to-text).

## 7. Embeddings for RAG

In [ ]:
emb = client.embeddings.create(
    model="qwen3-embedding",
    input=[
        "Retrieval-augmented generation pairs a vector store with an LLM.",
        "LoRA inserts low-rank adapter matrices into a frozen base model.",
    ],
)
print(f"dim={len(emb.data[0].embedding)}, first 5={emb.data[0].embedding[:5]}")

These vectors plug into any vector DB (Chroma, Qdrant, pgvector, etc.). Do not call `qwen3-embedding` for chat completions — it's not a chat model.

## 8. Cache isolation for sensitive prompts

NRP caches prompts and responses across users sharing a tenant. If your prompts could be sensitive, pass a `cache_salt` (≥256 random bits, base64-encoded) that only you know:

In [ ]:
import base64, os

project_salt = base64.b64encode(os.urandom(32)).decode("ascii")

completion = client.chat.completions.create(
    model="gpt-oss",
    messages=[{"role": "user", "content": "Summarize this internal memo: ..."}],
    extra_body={"cache_salt": project_salt},
)
print(completion.choices[0].message.content)

## Where to go next

- [Available models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) — full per-model cards and use-case recommendations.
- [Chat interfaces](https://nrp.ai/documentation/userdocs/ai/llm-managed/chat-interfaces/) and [coding CLI configs](https://nrp.ai/documentation/userdocs/ai/llm-managed/client-configs/) if you'd rather not script.
- [NRP card](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/NRP-Nautilus.html) — fair-use limits, cluster policies, where it fits alongside CHTC / UW Cloud.

## Comments